In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/grontho/bert-model/News-Categories-preprocessed.csv


In [4]:
# ============================================================
# DATASET 2 - NEWS CATEGORIES ONLY
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "bert-base-uncased"

LEARNING_RATE = 0.00003
BATCH_SIZE = 16
WEIGHT_DECAY = 0.1

EPOCHS = 5
WARMUP_RATIO = 0.1
MAX_LEN = 128
DROPOUT = 0.1

SEED = 42


# ============================================================
# 2. DATASET PATH
# ============================================================

DATA_DIR = "/kaggle/input/datasets/grontho/bert-model"

NEWS_FILE = os.path.join(
    DATA_DIR,
    "News-Categories-preprocessed.csv"
)

# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 4. SEED
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 5. DATASET CLASS
# ============================================================

class TextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len=MAX_LEN
    ):

        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len
        )

        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item


# ============================================================
# 6. METRICS
# ============================================================

def compute_metrics(pred):

    labels = pred.label_ids

    predictions = pred.predictions

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    preds = np.argmax(
        predictions,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="weighted",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# 7. LOAD DATASET
# ============================================================

print("\n" + "=" * 70)
print("Running BERT on: Dataset2_NewsCategories")
print("=" * 70)

df = pd.read_csv(NEWS_FILE)

print("Original shape:", df.shape)


# ============================================================
# 8. REMOVE MISSING VALUES
# ============================================================

df = df.dropna(
    subset=[
        "clean_text",
        "Category"
    ]
).reset_index(drop=True)

print("After removing NaN:", df.shape)


# ============================================================
# 9. LABEL ENCODING
# ============================================================

labels_unique = sorted(
    df["Category"].unique()
)

label2id = {
    label: idx
    for idx, label in enumerate(labels_unique)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

df["label_id"] = df["Category"].map(
    label2id
)

num_labels = len(labels_unique)

print("Number of classes:", num_labels)
print("Classes:", labels_unique)


# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

train_texts, test_texts, train_labels, test_labels = (
    train_test_split(
        df["clean_text"].tolist(),
        df["label_id"].tolist(),
        test_size=0.2,
        random_state=SEED,
        stratify=df["label_id"]
    )
)

print(
    "Training samples:",
    len(train_texts)
)

print(
    "Testing samples:",
    len(test_texts)
)


# ============================================================
# 11. TOKENIZER
# ============================================================

tokenizer = BertTokenizerFast.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 12. CREATE DATASETS
# ============================================================

train_dataset = TextDataset(
    train_texts,
    train_labels,
    tokenizer,
    MAX_LEN
)

test_dataset = TextDataset(
    test_texts,
    test_labels,
    tokenizer,
    MAX_LEN
)


# ============================================================
# 13. BERT MODEL
# ============================================================

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT
)

model.to(device)


# ============================================================
# 14. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="/kaggle/working/Dataset2_NewsCategories",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",

    save_strategy="epoch",
    save_total_limit=1,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,

    optim="adamw_torch",

    seed=SEED,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2
)


# ============================================================
# 15. TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)


# ============================================================
# 16. TRAINING
# ============================================================

print("\nStarting training...")

trainer.train()


# ============================================================
# 17. EVALUATION
# ============================================================

print("\nEvaluating...")

metrics = trainer.evaluate()


# ============================================================
# 18. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("DATASET 2 - NEWS CATEGORIES RESULTS")
print("=" * 70)

print(f"Accuracy  : {metrics['eval_accuracy'] * 100:.2f}%")
print(f"Precision : {metrics['eval_precision'] * 100:.2f}%")
print(f"Recall    : {metrics['eval_recall'] * 100:.2f}%")
print(f"F1 Score  : {metrics['eval_f1'] * 100:.2f}%")

Device: cuda
GPU: Tesla T4

Running BERT on: Dataset2_NewsCategories
Original shape: (252, 2)
After removing NaN: (252, 2)
Number of classes: 8
Classes: ['Artificial Intelligence ', 'Economy', 'Entertainment', 'Food', 'Health', 'International relations', 'Politics', 'Sports']
Training samples: 201
Testing samples: 51


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



DATASET 2 - NEWS CATEGORIES RESULTS
Accuracy  : 66.67%
Precision : 69.39%
Recall    : 66.67%
F1 Score  : 62.92%
